In [0]:
from pyspark.sql import functions as F

silver = "workspace.silver.adidas_us_sales"
dimension_gold = "workspace.gold.dim_ciudades"
checkpoint = "/Volumes/workspace/gold/dim_ciudades/checkpoint"

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.gold.dim_ciudades")


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold_examen")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {dimension_gold} (
        id_ciudad STRING, nombre_ciudad STRING, estado_ciudad STRING,
        region_ciudad STRING, ciudad_estado STRING
    ) USING DELTA
""")

In [0]:
def cargar_dimension(lote, _):
    if lote.isEmpty():
        return

    dimension = lote.select(
        F.sha2(F.to_json(F.struct("ciudad", "estado", "region")), 256).alias("id_ciudad"),
        F.col("ciudad").alias("nombre_ciudad"),
        F.col("estado").alias("estado_ciudad"),
        F.col("region").alias("region_ciudad"),
        F.concat_ws(", ", "ciudad", "estado").alias("ciudad_estado"),
    ).distinct()
    dimension.createOrReplaceTempView("miembros_dimension")
    lote.sparkSession.sql(f"""
        MERGE INTO {dimension_gold} AS destino
        USING miembros_dimension AS origen
        ON destino.id_ciudad = origen.id_ciudad
        WHEN NOT MATCHED THEN INSERT *
    """)

In [0]:
flujo = (
    spark.readStream.option("readChangeFeed", "true").table(silver)
    .filter("_change_type IN ('insert', 'update_postimage')")
    .writeStream.option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .foreachBatch(cargar_dimension)
    .start()
)
flujo.awaitTermination()
